# Organized research notebook

Cell source is retained from the supplied research archive. Saved outputs and credentials have been removed. Run the setup cell first. Check model, SAE, file paths, and run settings in the cells before starting a GPU experiment. This notebook has been checked statically, not rerun against the paper results. See `docs/RELEASE_REVIEW.md` for unresolved items.


Lexical-control workflow: first run `01_prepare_lexical_control.ipynb`, then `02_evaluate_lexical_control.ipynb`. The evaluator requires generated selected-feature CSVs and trained steering checkpoints. A fresh kernel is required for the evaluator's cache setup; the repository setup cell imports only standard-library modules. See `docs/LEXICAL_CONTROL.md` for parameters and resume behavior.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "src" / "sae_emotion" / "__init__.py").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the sae-emotion repository.")
sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)


# Lexical-shortcut control evaluation with resume support

This notebook evaluates the existing best SAE-steered checkpoints on a new
lexically filtered dataset.

It keeps fixed:

- the originally selected SAE features;
- the saved best steering checkpoint;
- the original zero-shot prompt:
  `In this text: {text}, the emotion implied is:`.

It changes only the evaluation dataset. For each model, it reports vanilla and
steered performance on the same examples.

Resume behavior:

- existing completed models are loaded from the saved CSV files and skipped;
- a model is complete only when both vanilla and steered overall results and
  all six per-emotion result rows exist;
- partial results from interrupted models are discarded and rerun;
- previously completed rows are preserved when new results are saved.


## Required startup order

Restart the kernel, then run all cells from the top. The first code cell must
execute before any import of `huggingface_hub`, `transformers`, `sae_lens`, or
`feature_steering_utils`.


In [ ]:
from __future__ import annotations

# IMPORTANT:
# Restart the kernel before running this notebook.
# This must be the first executable cell because Hugging Face cache paths
# are resolved when huggingface_hub / transformers / sae_lens are imported.

import os
import gc
import traceback
from dataclasses import dataclass
from typing import Dict, List, Optional

# ============================================================
# Hugging Face cache
# ============================================================

# This is the cache that previously contained the downloaded Gemma-3 SAEs.
HF_HOME = os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
HF_HUB_CACHE = f"{HF_HOME}/hub"

os.environ["HF_HOME"] = HF_HOME
os.environ["HF_HUB_CACHE"] = HF_HUB_CACHE

# Compatibility with packages that still inspect older variable names.
os.environ["HUGGINGFACE_HUB_CACHE"] = HF_HUB_CACHE
os.environ["HF_XET_CACHE"] = f"{HF_HOME}/xet"
os.environ["HF_DATASETS_CACHE"] = f"{HF_HOME}/datasets"
os.environ["TRANSFORMERS_CACHE"] = HF_HUB_CACHE
os.environ["TORCH_HOME"] = f"{HF_HOME}/torch"

# Do not hardcode an HF token in this notebook.
# It will use either:
#   1. the existing `HF_TOKEN` environment variable, or
#   2. the token saved by `hf auth login` under HF_HOME.
#
# Do not set HF_HUB_OFFLINE here because SAELens may still need repository
# metadata even when the SAE weights are already cached.


# ============================================================
# Imports that may initialize Hugging Face
# ============================================================

from huggingface_hub import constants

actual_hub_cache = os.path.abspath(constants.HF_HUB_CACHE)
expected_hub_cache = os.path.abspath(HF_HUB_CACHE)

print("Configured HF_HOME:", os.environ.get("HF_HOME"))
print("Configured HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE"))
print("Actual Hugging Face hub cache:", actual_hub_cache)

if actual_hub_cache != expected_hub_cache:
    raise RuntimeError(
        "Hugging Face was imported before the cache was configured. "
        "Restart the kernel and run this notebook from the first cell. "
        f"Expected {expected_hub_cache}, but got {actual_hub_cache}."
    )

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, f1_score


from huggingface_hub import hf_hub_download
from safetensors import safe_open
import sae_lens.loading.pretrained_sae_loaders as sae_loaders


def get_safetensors_tensor_shapes_from_cache(
    repo_id: str,
    filename: str,
):
    """
    Resolve the already cached Gemma Scope 2 safetensors file and
    inspect its metadata locally, avoiding SAELens's remote range request.
    """
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_files_only=False,
    )

    with safe_open(
        local_path,
        framework="pt",
        device="cpu",
    ) as f:
        return {
            tensor_name: list(
                f.get_slice(tensor_name).get_shape()
            )
            for tensor_name in f.keys()
        }


sae_loaders.get_safetensors_tensor_shapes = (
    get_safetensors_tensor_shapes_from_cache
)

print(
    "Patched Gemma Scope 2 shape lookup "
    "to use cached safetensors files."
)

from sae_emotion.feature_steering_utils import (
    DEFAULT_EMOTIONS,
    SAEFeatureFinetuner,
    prepare_emotion_dataframe,
    safe_save_csv,
    zero_shot_template,
)

print("Configuration and imports loaded successfully.")


# ============================================================
# User settings
# ============================================================

# Change only this block if the filtered dataset has a different path
# or different column names.
TEST_DATASET = {
    "name": "lexical_filtered_balanced",
    "path": (
        "data/lexical_shortcut_control/"
        "balanced_dataset_without_global_lexical_shortcuts.csv"
    ),
    "text_col": "text",
    "label_col": "emotion",
    "sep": ",",
    "drop_labels": None,
}

OUTPUT_DIR = "data/lexical_shortcut_steering_evaluation"

FEATURE_CSV_NAME = (
    "auto_selected_unique_features_with_affected_emotions.csv"
)

# Must match the directory containing the trained steering checkpoint.
STEERING_RUN_DIR_NAME = "lr_5e-3_l1_0p0"

CACHE_DIR: Optional[str] = None
DEVICE = "cuda"
DEVICE_MAP = "cuda"
TORCH_DTYPE = torch.bfloat16
EVAL_BATCH_SIZE = 16
MAX_LENGTH = 512

SAVE_PREDICTIONS = True

# Resume from existing output CSVs and skip fully completed models.
RESUME = True

# Add model names here only when you intentionally want to rerun them.
# Example: {"google/gemma-2-2b"}
FORCE_RERUN_MODELS = set()


@dataclass(frozen=True)
class ModelSpec:
    model_name: str
    feature_dir: str
    sae_size: Optional[str]

    @property
    def feature_csv_path(self) -> str:
        return os.path.join(
            self.feature_dir,
            FEATURE_CSV_NAME,
        )

    @property
    def steering_run_dir(self) -> str:
        return os.path.join(
            self.feature_dir,
            "global_feature_steering",
            STEERING_RUN_DIR_NAME,
        )

    @property
    def best_checkpoint_path(self) -> str:
        return os.path.join(
            self.steering_run_dir,
            "best_feature_steering.pt",
        )


MODEL_SPECS = [
    ModelSpec(
        model_name="google/gemma-2-2b",
        feature_dir="data/google-gemma-2-2b-feature-act",
        sae_size=None,
    ),
    ModelSpec(
        model_name="google/gemma-2-9b",
        feature_dir="data/google-gemma-2-9b-feature-act",
        sae_size=None,
    ),
    ModelSpec(
        model_name="google/gemma-3-4b-pt",
        feature_dir="data/google-gemma-3-4b-pt-feature-act-small",
        sae_size="small",
    ),
    ModelSpec(
        model_name="google/gemma-3-4b-it",
        feature_dir="data/google-gemma-3-4b-it-feature-act-small",
        sae_size="small",
    ),
    ModelSpec(
        model_name="google/gemma-3-12b-pt",
        feature_dir="data/google-gemma-3-12b-pt-feature-act-small",
        sae_size="small",
    ),
    ModelSpec(
        model_name="google/gemma-3-12b-it",
        feature_dir="data/google-gemma-3-12b-it-feature-act-small",
        sae_size="small",
    ),
    ModelSpec(
        model_name="meta-llama/Llama-3.1-8B",
        feature_dir="data/meta-llama-Llama-3.1-8B-feature-act",
        sae_size=None,
    ),
]


In [ ]:
# ============================================================
# Dataset, metric, saving, and resume helpers
# ============================================================

OVERALL_RESULTS_FILENAME = "lexical_control_results_long.csv"
PER_EMOTION_RESULTS_FILENAME = "lexical_control_per_emotion_long.csv"
PREDICTIONS_FILENAME = "lexical_control_predictions.csv"


def result_path(filename: str) -> str:
    return os.path.join(OUTPUT_DIR, filename)


def validate_required_files(model_spec: ModelSpec) -> None:
    required_paths = [
        TEST_DATASET["path"],
        model_spec.feature_csv_path,
        model_spec.best_checkpoint_path,
    ]

    missing = [
        path
        for path in required_paths
        if not os.path.exists(path)
    ]

    if missing:
        formatted = "\n".join(
            f"  - {path}"
            for path in missing
        )
        raise FileNotFoundError(
            f"Missing required files for "
            f"{model_spec.model_name}:\n{formatted}"
        )


def load_filtered_test_dataframe() -> pd.DataFrame:
    """
    Load raw text and gold labels without applying any prompt yet.
    """
    df = prepare_emotion_dataframe(
        csv_path=TEST_DATASET["path"],
        text_col=TEST_DATASET["text_col"],
        label_col=TEST_DATASET["label_col"],
        sep=TEST_DATASET["sep"],
        apply_prompt_template=False,
        keep_emotions=list(DEFAULT_EMOTIONS),
        drop_labels=TEST_DATASET["drop_labels"],
    )

    df = df.rename(
        columns={"input_text": "raw_text"}
    )

    return df[
        ["raw_text", "emotion"]
    ].reset_index(drop=True)


def apply_original_prompt(
    raw_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Apply the exact original prompt used by feature_steering_utils:
        In this text: {text}, the emotion implied is:
    """
    prompted_df = raw_df.copy()
    prompted_df["input_text"] = (
        prompted_df["raw_text"]
        .astype(str)
        .map(zero_shot_template)
    )

    return prompted_df[
        ["raw_text", "input_text", "emotion"]
    ]


def calculate_metrics(
    y_true: List[str],
    y_pred: List[str],
    emotions: List[str],
) -> tuple[Dict[str, float], List[Dict[str, float]]]:
    report = classification_report(
        y_true,
        y_pred,
        labels=emotions,
        output_dict=True,
        zero_division=0,
    )

    overall = {
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=emotions,
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=emotions,
                average="weighted",
                zero_division=0,
            )
        ),
    }

    per_emotion = []

    for emotion in emotions:
        emotion_metrics = report.get(emotion, {})

        per_emotion.append(
            {
                "emotion": emotion,
                "precision": float(
                    emotion_metrics.get(
                        "precision",
                        0.0,
                    )
                ),
                "recall": float(
                    emotion_metrics.get(
                        "recall",
                        0.0,
                    )
                ),
                "f1": float(
                    emotion_metrics.get(
                        "f1-score",
                        0.0,
                    )
                ),
                "support": int(
                    emotion_metrics.get(
                        "support",
                        0,
                    )
                ),
            }
        )

    return overall, per_emotion


def add_predictions(
    prediction_rows: List[Dict],
    prompted_df: pd.DataFrame,
    logits: np.ndarray,
    predictions: List[str],
    model_name: str,
    condition: str,
    emotions: List[str],
) -> None:
    for row_idx, row in (
        prompted_df
        .reset_index(drop=True)
        .iterrows()
    ):
        output_row = {
            "model": model_name,
            "dataset": TEST_DATASET["name"],
            "prompt": "original",
            "condition": condition,
            "example_id": int(row_idx),
            "raw_text": row["raw_text"],
            "prompted_text": row["input_text"],
            "gold_emotion": row["emotion"],
            "pred_emotion": predictions[row_idx],
        }

        for emotion_idx, emotion in enumerate(emotions):
            output_row[f"logit_{emotion}"] = float(
                logits[row_idx, emotion_idx]
            )

        prediction_rows.append(output_row)


def deduplicate_result_rows(
    rows: List[Dict],
    subset: List[str],
) -> List[Dict]:
    if not rows:
        return []

    df = pd.DataFrame(rows)
    df = df.drop_duplicates(
        subset=subset,
        keep="last",
    )
    return df.to_dict("records")


def save_current_results(
    overall_rows: List[Dict],
    per_emotion_rows: List[Dict],
    prediction_rows: List[Dict],
) -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    overall_rows[:] = deduplicate_result_rows(
        overall_rows,
        subset=[
            "model",
            "dataset",
            "prompt",
            "condition",
        ],
    )

    per_emotion_rows[:] = deduplicate_result_rows(
        per_emotion_rows,
        subset=[
            "model",
            "dataset",
            "prompt",
            "condition",
            "emotion",
        ],
    )

    if SAVE_PREDICTIONS:
        prediction_rows[:] = deduplicate_result_rows(
            prediction_rows,
            subset=[
                "model",
                "dataset",
                "prompt",
                "condition",
                "example_id",
            ],
        )

    if overall_rows:
        safe_save_csv(
            pd.DataFrame(overall_rows),
            result_path(OVERALL_RESULTS_FILENAME),
        )

    if per_emotion_rows:
        safe_save_csv(
            pd.DataFrame(per_emotion_rows),
            result_path(PER_EMOTION_RESULTS_FILENAME),
        )

    if SAVE_PREDICTIONS and prediction_rows:
        safe_save_csv(
            pd.DataFrame(prediction_rows),
            result_path(PREDICTIONS_FILENAME),
        )


def find_completed_models(
    overall_df: pd.DataFrame,
    per_emotion_df: pd.DataFrame,
) -> set[str]:
    """
    A model is complete only when:
      1. vanilla and steered overall rows exist; and
      2. vanilla and steered per-emotion rows exist for every emotion.
    """
    if overall_df.empty or per_emotion_df.empty:
        return set()

    required_conditions = {"vanilla", "steered"}
    required_emotions = set(DEFAULT_EMOTIONS)

    candidate_models = set(overall_df["model"].dropna())
    completed_models = set()

    for model_name in candidate_models:
        model_overall = overall_df[
            overall_df["model"] == model_name
        ]
        overall_conditions = set(
            model_overall["condition"].dropna()
        )

        if not required_conditions.issubset(
            overall_conditions
        ):
            continue

        model_per_emotion = per_emotion_df[
            per_emotion_df["model"] == model_name
        ]

        per_emotion_complete = True
        for condition in required_conditions:
            condition_emotions = set(
                model_per_emotion.loc[
                    model_per_emotion["condition"] == condition,
                    "emotion",
                ].dropna()
            )

            if not required_emotions.issubset(
                condition_emotions
            ):
                per_emotion_complete = False
                break

        if per_emotion_complete:
            completed_models.add(model_name)

    return completed_models


def load_resume_state() -> tuple[
    List[Dict],
    List[Dict],
    List[Dict],
    set[str],
]:
    """
    Load completed results from disk.

    Rows belonging to incomplete models are intentionally discarded so those
    models can be rerun cleanly. Models listed in FORCE_RERUN_MODELS are also
    removed from the loaded state.
    """
    if not RESUME:
        print("Resume disabled. Starting with empty result tables.")
        return [], [], [], set()

    overall_path = result_path(
        OVERALL_RESULTS_FILENAME
    )
    per_emotion_path = result_path(
        PER_EMOTION_RESULTS_FILENAME
    )
    prediction_path = result_path(
        PREDICTIONS_FILENAME
    )

    if not os.path.exists(overall_path):
        print("No previous overall results found. Starting fresh.")
        return [], [], [], set()

    if not os.path.exists(per_emotion_path):
        print(
            "Previous overall results exist, but per-emotion results "
            "are missing. Starting fresh so completion can be verified."
        )
        return [], [], [], set()

    overall_df = pd.read_csv(overall_path)
    per_emotion_df = pd.read_csv(per_emotion_path)

    completed_models = find_completed_models(
        overall_df=overall_df,
        per_emotion_df=per_emotion_df,
    )

    completed_models -= set(FORCE_RERUN_MODELS)

    overall_df = overall_df[
        overall_df["model"].isin(completed_models)
    ].copy()

    per_emotion_df = per_emotion_df[
        per_emotion_df["model"].isin(completed_models)
    ].copy()

    if SAVE_PREDICTIONS and os.path.exists(prediction_path):
        prediction_df = pd.read_csv(prediction_path)
        prediction_df = prediction_df[
            prediction_df["model"].isin(completed_models)
        ].copy()
    else:
        prediction_df = pd.DataFrame()

    if completed_models:
        print("Completed models loaded and skipped:")
        for model_name in sorted(completed_models):
            print(f"  - {model_name}")
    else:
        print("No fully completed models found. All models will run.")

    if FORCE_RERUN_MODELS:
        print("Models forced to rerun:")
        for model_name in sorted(FORCE_RERUN_MODELS):
            print(f"  - {model_name}")

    return (
        overall_df.to_dict("records"),
        per_emotion_df.to_dict("records"),
        prediction_df.to_dict("records"),
        completed_models,
    )


def create_comparison_tables(
    overall_df: pd.DataFrame,
    per_emotion_df: pd.DataFrame,
) -> None:
    metric_columns = [
        "accuracy",
        "macro_f1",
        "weighted_f1",
    ]

    # Only completed condition pairs can be pivoted safely.
    completed_models = find_completed_models(
        overall_df=overall_df,
        per_emotion_df=per_emotion_df,
    )

    overall_df = overall_df[
        overall_df["model"].isin(completed_models)
    ].copy()
    per_emotion_df = per_emotion_df[
        per_emotion_df["model"].isin(completed_models)
    ].copy()

    if overall_df.empty or per_emotion_df.empty:
        print(
            "No complete model results are available for comparison tables."
        )
        return

    overall_comparison = overall_df.pivot(
        index=[
            "model",
            "dataset",
            "prompt",
            "n",
        ],
        columns="condition",
        values=metric_columns,
    )

    overall_comparison.columns = [
        f"{metric}_{condition}"
        for metric, condition
        in overall_comparison.columns
    ]

    overall_comparison = (
        overall_comparison
        .reset_index()
    )

    for metric in metric_columns:
        overall_comparison[f"delta_{metric}"] = (
            overall_comparison[
                f"{metric}_steered"
            ]
            - overall_comparison[
                f"{metric}_vanilla"
            ]
        )

    safe_save_csv(
        overall_comparison,
        os.path.join(
            OUTPUT_DIR,
            "lexical_control_comparison.csv",
        ),
    )

    per_emotion_comparison = per_emotion_df.pivot(
        index=[
            "model",
            "dataset",
            "prompt",
            "emotion",
            "support",
        ],
        columns="condition",
        values=[
            "precision",
            "recall",
            "f1",
        ],
    )

    per_emotion_comparison.columns = [
        f"{metric}_{condition}"
        for metric, condition
        in per_emotion_comparison.columns
    ]

    per_emotion_comparison = (
        per_emotion_comparison
        .reset_index()
    )

    per_emotion_comparison["delta_f1"] = (
        per_emotion_comparison["f1_steered"]
        - per_emotion_comparison["f1_vanilla"]
    )

    safe_save_csv(
        per_emotion_comparison,
        os.path.join(
            OUTPUT_DIR,
            "lexical_control_per_emotion_comparison.csv",
        ),
    )


def cleanup_after_model(
    finetuner: Optional[SAEFeatureFinetuner],
) -> None:
    if finetuner is not None:
        del finetuner

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


In [ ]:
# ============================================================
# Evaluation
# ============================================================


def evaluate_model(
    model_spec: ModelSpec,
    prompted_df: pd.DataFrame,
    overall_rows: List[Dict],
    per_emotion_rows: List[Dict],
    prediction_rows: List[Dict],
) -> None:
    """
    Evaluate one model.

    Results are first accumulated in model-local lists. They are added to the
    global result lists only after both vanilla and steered evaluations finish
    successfully. This prevents failed runs from saving partial model rows.
    """
    validate_required_files(model_spec)

    print("=" * 100)
    print(f"Model: {model_spec.model_name}")
    print(f"Feature CSV: {model_spec.feature_csv_path}")
    print(
        f"Best checkpoint: "
        f"{model_spec.best_checkpoint_path}"
    )
    print("=" * 100)

    finetuner = None
    model_overall_rows: List[Dict] = []
    model_per_emotion_rows: List[Dict] = []
    model_prediction_rows: List[Dict] = []

    try:
        finetuner = SAEFeatureFinetuner(
            model_name=model_spec.model_name,
            feature_csv_path=model_spec.feature_csv_path,
            emotions=list(DEFAULT_EMOTIONS),
            sae_size=model_spec.sae_size,
            cache_dir=CACHE_DIR,
            device=DEVICE,
            device_map=DEVICE_MAP,
            torch_dtype=TORCH_DTYPE,
            max_length=MAX_LENGTH,
        )

        # ----------------------------------------------------
        # Vanilla model
        # ----------------------------------------------------
        (
            y_true,
            vanilla_pred,
            vanilla_logits,
        ) = finetuner.predict_clean(
            prompted_df,
            batch_size=EVAL_BATCH_SIZE,
        )

        (
            vanilla_metrics,
            vanilla_per_emotion,
        ) = calculate_metrics(
            y_true,
            vanilla_pred,
            finetuner.emotions,
        )

        model_overall_rows.append(
            {
                "model": model_spec.model_name,
                "dataset": TEST_DATASET["name"],
                "prompt": "original",
                "condition": "vanilla",
                "n": int(len(prompted_df)),
                **vanilla_metrics,
            }
        )

        for row in vanilla_per_emotion:
            model_per_emotion_rows.append(
                {
                    "model": model_spec.model_name,
                    "dataset": TEST_DATASET["name"],
                    "prompt": "original",
                    "condition": "vanilla",
                    **row,
                }
            )

        if SAVE_PREDICTIONS:
            add_predictions(
                prediction_rows=model_prediction_rows,
                prompted_df=prompted_df,
                logits=vanilla_logits,
                predictions=vanilla_pred,
                model_name=model_spec.model_name,
                condition="vanilla",
                emotions=finetuner.emotions,
            )

        # ----------------------------------------------------
        # Existing best steered model
        # ----------------------------------------------------
        finetuner.load_state(
            model_spec.best_checkpoint_path
        )

        (
            y_true_steered,
            steered_pred,
            steered_logits,
        ) = finetuner.predict(
            prompted_df,
            batch_size=EVAL_BATCH_SIZE,
        )

        if y_true_steered != y_true:
            raise RuntimeError(
                "Vanilla and steered label ordering differs."
            )

        (
            steered_metrics,
            steered_per_emotion,
        ) = calculate_metrics(
            y_true_steered,
            steered_pred,
            finetuner.emotions,
        )

        model_overall_rows.append(
            {
                "model": model_spec.model_name,
                "dataset": TEST_DATASET["name"],
                "prompt": "original",
                "condition": "steered",
                "n": int(len(prompted_df)),
                **steered_metrics,
            }
        )

        for row in steered_per_emotion:
            model_per_emotion_rows.append(
                {
                    "model": model_spec.model_name,
                    "dataset": TEST_DATASET["name"],
                    "prompt": "original",
                    "condition": "steered",
                    **row,
                }
            )

        if SAVE_PREDICTIONS:
            add_predictions(
                prediction_rows=model_prediction_rows,
                prompted_df=prompted_df,
                logits=steered_logits,
                predictions=steered_pred,
                model_name=model_spec.model_name,
                condition="steered",
                emotions=finetuner.emotions,
            )

        delta_macro_f1 = (
            steered_metrics["macro_f1"]
            - vanilla_metrics["macro_f1"]
        )

        print(
            f"Vanilla macro-F1: "
            f"{vanilla_metrics['macro_f1']:.4f}"
        )
        print(
            f"Steered macro-F1: "
            f"{steered_metrics['macro_f1']:.4f}"
        )
        print(
            f"Delta macro-F1: "
            f"{delta_macro_f1:+.4f}"
        )

        # Commit this model only after the full evaluation succeeds.
        overall_rows.extend(model_overall_rows)
        per_emotion_rows.extend(model_per_emotion_rows)
        prediction_rows.extend(model_prediction_rows)

        save_current_results(
            overall_rows,
            per_emotion_rows,
            prediction_rows,
        )

        print(
            f"Saved completed results for "
            f"{model_spec.model_name}."
        )

    finally:
        cleanup_after_model(finetuner)


def main() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    raw_test_df = load_filtered_test_dataframe()
    prompted_df = apply_original_prompt(raw_test_df)

    safe_save_csv(
        raw_test_df,
        os.path.join(
            OUTPUT_DIR,
            "lexical_filtered_raw_test_df.csv",
        ),
    )

    safe_save_csv(
        prompted_df,
        os.path.join(
            OUTPUT_DIR,
            "lexical_filtered_prompted_test_df.csv",
        ),
    )

    print(f"Dataset: {TEST_DATASET['name']}")
    print(f"Path: {TEST_DATASET['path']}")
    print(f"Number of examples: {len(prompted_df)}")
    print("Label counts:")
    print(
        prompted_df["emotion"]
        .value_counts()
        .reindex(DEFAULT_EMOTIONS)
    )
    print()
    print("Prompt example:")
    print(prompted_df.iloc[0]["input_text"])
    print()

    (
        overall_rows,
        per_emotion_rows,
        prediction_rows,
        completed_models,
    ) = load_resume_state()

    failures: List[str] = []

    models_to_run = [
        model_spec
        for model_spec in MODEL_SPECS
        if model_spec.model_name not in completed_models
    ]

    print(
        f"\nModels completed: {len(completed_models)} / "
        f"{len(MODEL_SPECS)}"
    )
    print(f"Models remaining: {len(models_to_run)}")

    if models_to_run:
        print("Models scheduled for this run:")
        for model_spec in models_to_run:
            print(f"  - {model_spec.model_name}")
    else:
        print("All configured models are already complete.")

    for model_spec in models_to_run:
        try:
            evaluate_model(
                model_spec=model_spec,
                prompted_df=prompted_df,
                overall_rows=overall_rows,
                per_emotion_rows=per_emotion_rows,
                prediction_rows=prediction_rows,
            )
        except Exception as exc:
            message = (
                f"FAILED {model_spec.model_name}: "
                f"{exc!r}"
            )
            print(message)
            traceback.print_exc()
            failures.append(message)
            cleanup_after_model(None)

    save_current_results(
        overall_rows,
        per_emotion_rows,
        prediction_rows,
    )

    if overall_rows and per_emotion_rows:
        overall_df = pd.DataFrame(overall_rows)
        per_emotion_df = pd.DataFrame(
            per_emotion_rows
        )

        create_comparison_tables(
            overall_df,
            per_emotion_df,
        )

    final_completed_models = find_completed_models(
        overall_df=pd.DataFrame(overall_rows),
        per_emotion_df=pd.DataFrame(per_emotion_rows),
    ) if overall_rows and per_emotion_rows else set()

    print("\nSaved outputs to:")
    print(OUTPUT_DIR)
    print(
        f"Completed models after this run: "
        f"{len(final_completed_models)} / {len(MODEL_SPECS)}"
    )

    if failures:
        print("\nSome model evaluations failed:")
        for failure in failures:
            print(f"- {failure}")
    else:
        print(
            "\nAll scheduled model evaluations "
            "finished successfully."
        )


main()


In [ ]:
# ============================================================
# Display compact F1 result tables after evaluation
# ============================================================

overall_path = os.path.join(
    OUTPUT_DIR,
    "lexical_control_comparison.csv",
)

per_emotion_path = os.path.join(
    OUTPUT_DIR,
    "lexical_control_per_emotion_comparison.csv",
)

# Overall macro-F1
if os.path.exists(overall_path):
    overall_table = pd.read_csv(overall_path)

    display_columns = [
        "model",
        "n",
        "macro_f1_vanilla",
        "macro_f1_steered",
        "delta_macro_f1",
    ]

    available_columns = [
        column
        for column in display_columns
        if column in overall_table.columns
    ]

    display(
        overall_table[available_columns]
        .sort_values("model")
        .reset_index(drop=True)
    )
else:
    print("Overall comparison table has not been created yet.")


# Per-emotion F1 improvement
if os.path.exists(per_emotion_path):
    per_emotion_table = pd.read_csv(
        per_emotion_path
    )

    f1_table = per_emotion_table.pivot(
        index="model",
        columns="emotion",
        values="delta_f1",
    )

    f1_table = f1_table.reindex(
        columns=list(DEFAULT_EMOTIONS)
    )

    display(
        f1_table
        .reset_index()
        .sort_values("model")
        .reset_index(drop=True)
    )
else:
    print("Per-emotion comparison table has not been created yet.")